In [109]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.13.0
tiktoken version: 0.13.0


In [110]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [111]:
integers = tokenizer.encode("Hello, world!")
print("Encoded integers:", integers)

Encoded integers: [15496, 11, 995, 0]


In [112]:
for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")

15496 -> Hello
11 -> ,
995 ->  world
0 -> !


In [113]:
tokenizer.encode("Hello")
tokenizer.encode(",")
tokenizer.encode("world")
tokenizer.encode("!")

[0]

In [114]:
tokenizer.decode([15496, 11, 995, 0])

'Hello, world!'

Exercise 2.2

In [115]:
import torch
import tiktoken
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self,txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids)-max_length, stride):
            input_trunk = token_ids[i:i+max_length]
            target_trunk = token_ids[i+1:i+max_length+1]
            self.input_ids.append(torch.tensor(input_trunk))
            self.target_ids.append(torch.tensor(target_trunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader(txt, batch_size, max_length=256, stride=128):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    return dataloader


with open("../the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

In [116]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=2, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

tensor([[  26,  290],
        [ 843,  340],
        [ 523,  765],
        [3619,    0]])

In [117]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=8, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

tensor([[  531,   351, 27322,   540, 11293,    13,   366,   464],
        [  262,  6678, 40315, 10455,   546,   683,    11, 10597],
        [  262,  4112,   957,  1483,   286,  3619,   338,  2270],
        [  890,   276,   284,   910,    25,   366,  3856, 44455]])